:meth:`DesignConstraints.to_dict` exports the constraint set as a plain, JSON-serializable dictionary with one key per field plus ``parent``, so a design campaign can be written to disk, logged next to its results, or handed to another process. It takes no parameters, and :meth:`DesignConstraints.from_dict` reconstructs an equal object from its output.

In [1]:
import json

import pandas as pd
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=5)
seq = df_seq["sequence"].iloc[0]
tmd_start, tmd_stop = int(df_seq["tmd_start"].iloc[0]), int(df_seq["tmd_stop"].iloc[0])

dc = aa.DesignConstraints(immutable_positions=[tmd_start, tmd_stop],
                          permitted_substitutions=["A", "L", "V", "I"],
                          forbidden_substitutions={tmd_start + 6: ["V"]},
                          n_mut_max=2,
                          min_identity=0.95,
                          forbidden_motifs=["WW"],
                          parent=seq)
dict_constraints = dc.to_dict()
df_constraints = pd.DataFrame({"field": list(dict_constraints),
                               "value": [str(v)[:60] for v in dict_constraints.values()]})
aa.display_df(df_constraints, n_rows=10, show_shape=True)

DataFrame shape: (10, 2)


,field,value
1,immutable_positions,"[37, 59]"
2,mutable_positions,None
3,permitted_substitutions,"['A', 'L', 'V', 'I']"
4,forbidden_substitutions,{43: ['V']}
5,n_mut_max,2
6,min_identity,0.95
7,max_identity,None
8,forbidden_motifs,['WW']
9,required_motifs,None
10,parent,MQKVTLGLLVFLAGF...GVLCAMGIIIVMSAK


Unset fields are ``None``, and the containers are copies, so editing the exported dictionary never changes the object it came from. The dictionary survives a JSON round trip: `from_dict` converts the integer position keys that JSON turned into digit strings back.

In [2]:
text = json.dumps(dict_constraints)
dict_constraints["n_mut_max"] = 99           # editing the export leaves the object untouched
dc_json = aa.DesignConstraints.from_dict(dict_constraints=json.loads(text))

df_roundtrip = pd.DataFrame([dict(step="object", n_mut_max=dc.n_mut_max, json_chars=len(text)),
                             dict(step="edited export", n_mut_max=dict_constraints["n_mut_max"],
                                  json_chars=len(text)),
                             dict(step="from JSON", n_mut_max=dc_json.n_mut_max, json_chars=len(text))])
aa.display_df(df_roundtrip, n_rows=10, show_shape=True)
print("round trip equal:", dc_json == dc)

DataFrame shape: (3, 3)


,step,n_mut_max,json_chars
1,object,2,365
2,edited export,99,365
3,from JSON,2,365


round trip equal: True
